In [1]:
"""
Stage 3: Seasonality Detection
Comprehensive check for daily, weekly, and yearly seasonality patterns
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.graphics.tsaplots import plot_acf
from statsmodels.tsa.stattools import acf
from scipy import signal


In [2]:
df = pd.read_csv("../data/london_merged_weather_bikes.csv")  # adjust path and format


#TODO:
date_col = 'timestamp' 
df[date_col] = pd.to_datetime(df[date_col])

In [3]:
df.head(3)

,timestamp,cnt,t1,t2,hum,wind_speed,weather_code,is_holiday,is_weekend,season,temperature_c,humidity_percent,dew_point_c,rainfall_mm,snowfall_cm,wind_speed_ms,solar_radiation_wm2,solar_radiation_mjm2
0,2015-01-04 00:00:00,182.0,3.0,2.0,93.0,6.0,3.0,0.0,1.0,3.0,3.4,94,2.6,0.0,0.0,2.16,0.0,0.0
1,2015-01-04 01:00:00,138.0,3.0,2.5,93.0,5.0,1.0,0.0,1.0,3.0,2.8,96,2.2,0.0,0.0,2.01,0.0,0.0
2,2015-01-04 02:00:00,134.0,2.5,2.5,96.5,0.0,1.0,0.0,1.0,3.0,2.2,97,1.7,0.0,0.0,1.84,0.0,0.0


In [4]:
# Prepare data
df_ts = df.set_index(date_col).sort_index()
target = 'cnt'
y = df_ts[target]


In [5]:
print(y.shape)
print(y.dtype)
print(y.isna().sum(), "NaNs")
print(y.describe())
print(y.index[:3], y.index[-3:])

(17544,)
float64
130 NaNs
count    17414.000000
mean      1143.101642
std       1085.108068
min          0.000000
25%        257.000000
50%        844.000000
75%       1671.750000
max       7860.000000
Name: cnt, dtype: float64
DatetimeIndex(['2015-01-04 00:00:00', '2015-01-04 01:00:00',
               '2015-01-04 02:00:00'],
              dtype='datetime64[ns]', name='timestamp', freq=None) DatetimeIndex(['2017-01-03 21:00:00', '2017-01-03 22:00:00',
               '2017-01-03 23:00:00'],
              dtype='datetime64[ns]', name='timestamp', freq=None)


In [6]:
# interpolate missing values to find out ACF patterns without gaps
y = df_ts[target].interpolate(method='time')

In [7]:

print("="*70)
print("SEASONALITY DETECTION")
print("="*70)


SEASONALITY DETECTION


In [8]:

# ============================================================================
# 1. ACF PLOT - Visual inspection of periodic patterns
# ============================================================================
print("\n[1] ACF ANALYSIS")
print("-"*70)

fig, axes = plt.subplots(3, 1, figsize=(15, 12))

# ACF with different lag ranges to see different seasonalities
lag_configs = [
    (72, "ACF: First 3 days (72 hours) - Check daily pattern"),
    (336, "ACF: First 2 weeks (336 hours) - Check weekly pattern"),
    (1000, "ACF: First ~42 days (1000 hours) - Check longer patterns")
]

for idx, (lags, title) in enumerate(lag_configs):
    plot_acf(y, lags=lags, ax=axes[idx], alpha=0.05)
    axes[idx].set_title(title)
    axes[idx].set_xlabel('Lag (hours)')
    axes[idx].set_ylabel('Autocorrelation')

plt.tight_layout()
plt.savefig('img/seasonality_acf_london.png', dpi=300, bbox_inches='tight')
print("ACF plots saved as 'seasonality_acf_london.png'")
plt.close()



[1] ACF ANALYSIS
----------------------------------------------------------------------
ACF plots saved as 'seasonality_acf_london.png'


In [9]:

# ============================================================================
# 2. AUTOCORRELATION AT SPECIFIC LAGS
# ============================================================================
print("\n[2] AUTOCORRELATION AT KEY LAGS")
print("-"*70)

# Calculate autocorrelation for specific lags
specific_lags = {
    'Daily (24h)': 24,
    'Weekly (168h)': 168,
    'Bi-weekly (336h)': 336,
    'Monthly (~730h)': 730,
    'Yearly (8760h)': 8760
}

acf_values = acf(y, nlags=max(specific_lags.values()), fft=True)

print("\nAutocorrelation values at key lags:")
for name, lag in specific_lags.items():
    if lag < len(acf_values):
        print(f"  {name:20s} lag={lag:5d}  ACF={acf_values[lag]:.4f}")
    else:
        print(f"  {name:20s} lag={lag:5d}  [Not enough data]")



[2] AUTOCORRELATION AT KEY LAGS
----------------------------------------------------------------------

Autocorrelation values at key lags:
  Daily (24h)          lag=   24  ACF=0.8039
  Weekly (168h)        lag=  168  ACF=0.8869
  Bi-weekly (336h)     lag=  336  ACF=0.8706
  Monthly (~730h)      lag=  730  ACF=-0.2018
  Yearly (8760h)       lag= 8760  ACF=0.3873


In [10]:

# ============================================================================
# 3. VISUAL AGGREGATION
# ============================================================================
print("\n[3] VISUAL AGGREGATION - Average patterns")
print("-"*70)

# Extract time components
df_agg = df.copy()
df_agg['Hour'] = df_agg[date_col].dt.hour
df_agg['DayOfWeek'] = df_agg[date_col].dt.dayofweek  # 0=Monday, 6=Sunday
df_agg['Month'] = df_agg[date_col].dt.month

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 3a. Daily pattern: Average by hour
hourly_avg = df_agg.groupby('Hour')[target].agg(['mean', 'std'])
axes[0].plot(hourly_avg.index, hourly_avg['mean'], marker='o', linewidth=2)
axes[0].fill_between(hourly_avg.index, 
                      hourly_avg['mean'] - hourly_avg['std'],
                      hourly_avg['mean'] + hourly_avg['std'],
                      alpha=0.3)
axes[0].set_xlabel('Hour of Day')
axes[0].set_ylabel('Average Bike Count')
axes[0].set_title('Daily Pattern: Average by Hour')
axes[0].grid(True, alpha=0.3)
axes[0].set_xticks(range(0, 24, 2))

# 3b. Weekly pattern: Average by day of week
day_names = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
weekly_avg = df_agg.groupby('DayOfWeek')[target].agg(['mean', 'std'])
axes[1].bar(weekly_avg.index, weekly_avg['mean'], yerr=weekly_avg['std'], 
            capsize=5, alpha=0.7)
axes[1].set_xlabel('Day of Week')
axes[1].set_ylabel('Average Bike Count')
axes[1].set_title('Weekly Pattern: Average by Day')
axes[1].set_xticks(range(7))
axes[1].set_xticklabels(day_names)
axes[1].grid(True, alpha=0.3, axis='y')

# 3c. Yearly pattern: Average by month
month_names = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 
               'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
monthly_avg = df_agg.groupby('Month')[target].agg(['mean', 'std'])
axes[2].plot(monthly_avg.index, monthly_avg['mean'], marker='o', linewidth=2)
axes[2].fill_between(monthly_avg.index,
                      monthly_avg['mean'] - monthly_avg['std'],
                      monthly_avg['mean'] + monthly_avg['std'],
                      alpha=0.3)
axes[2].set_xlabel('Month')
axes[2].set_ylabel('Average Bike Count')
axes[2].set_title('Yearly Pattern: Average by Month')
axes[2].set_xticks(range(1, 13))
axes[2].set_xticklabels(month_names, rotation=45)
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('img/seasonality_visual_aggregation_london.png', dpi=300, bbox_inches='tight')
print("Visual aggregation saved as 'seasonality_visual_aggregation_london.png'")
plt.close()

# Print statistics
print("\nPattern strength (coefficient of variation):")
print(f"  Hourly:  CV = {hourly_avg['mean'].std() / hourly_avg['mean'].mean():.4f}")
print(f"  Weekly:  CV = {weekly_avg['mean'].std() / weekly_avg['mean'].mean():.4f}")
print(f"  Monthly: CV = {monthly_avg['mean'].std() / monthly_avg['mean'].mean():.4f}")



[3] VISUAL AGGREGATION - Average patterns
----------------------------------------------------------------------
Visual aggregation saved as 'seasonality_visual_aggregation_london.png'

Pattern strength (coefficient of variation):
  Hourly:  CV = 0.7558
  Weekly:  CV = 0.1062
  Monthly: CV = 0.2334


In [11]:

# ============================================================================
# 4. PERIODOGRAM - Frequency domain analysis
# ============================================================================
print("\n[4] PERIODOGRAM - Dominant frequency identification")
print("-"*70)

# Calculate periodogram
frequencies, power = signal.periodogram(y.values, fs=1.0)  # fs=1 for hourly data


# Find dominant frequencies (top 5 peaks)
peak_indices = signal.find_peaks(power, height=np.percentile(power, 95))[0]
peak_freqs = frequencies[peak_indices]
peak_powers = power[peak_indices]




[4] PERIODOGRAM - Dominant frequency identification
----------------------------------------------------------------------


In [12]:

# Sort by power
sorted_indices = np.argsort(peak_powers)[::-1][:5]
dominant_freqs = peak_freqs[sorted_indices]
dominant_powers = peak_powers[sorted_indices]


In [13]:


print("\nTop 5 dominant frequencies:")
for i, (freq, pwr) in enumerate(zip(dominant_freqs, dominant_powers), 1):
    if freq > 0:
        period = 1 / freq
        print(f"  {i}. Frequency: {freq:.6f} -> Period: {period:.2f} hours ({period/24:.2f} days)")
    else:
        print(f"  {i}. Frequency: {freq:.6f} (DC component)")



Top 5 dominant frequencies:
  1. Frequency: 0.041667 -> Period: 24.00 hours (1.00 days)
  2. Frequency: 0.083333 -> Period: 12.00 hours (0.50 days)
  3. Frequency: 0.125000 -> Period: 8.00 hours (0.33 days)
  4. Frequency: 0.208333 -> Period: 4.80 hours (0.20 days)
  5. Frequency: 0.000114 -> Period: 8772.00 hours (365.50 days)


In [14]:

# Plot periodogram
fig, axes = plt.subplots(2, 1, figsize=(15, 10))

# Full periodogram
axes[0].semilogy(frequencies[1:], power[1:])  # Skip DC component
axes[0].set_xlabel('Frequency (cycles per hour)')
axes[0].set_ylabel('Power Spectral Density')
axes[0].set_title('Periodogram: Full spectrum')
axes[0].grid(True, alpha=0.3)

# Zoomed to show low frequencies (long periods)
max_freq = 0.01  # Show up to 100 hour periods
mask = frequencies < max_freq
axes[1].semilogy(frequencies[mask], power[mask])
axes[1].set_xlabel('Frequency (cycles per hour)')
axes[1].set_ylabel('Power Spectral Density')
axes[1].set_title('Periodogram: Low frequencies (long periods)')
axes[1].grid(True, alpha=0.3)

# Mark expected seasonalities
expected_periods = [24, 168]  # daily, weekly
for period in expected_periods:
    freq = 1 / period
    if freq < max_freq:
        axes[1].axvline(freq, color='red', linestyle='--', alpha=0.5, 
                       label=f'{period}h period')
axes[1].legend()

plt.tight_layout()
plt.savefig('img/seasonality_periodogram_london.png', dpi=300, bbox_inches='tight')
print("Periodogram saved as 'seasonality_periodogram_london.png'")
plt.close()


Periodogram saved as 'seasonality_periodogram_london.png'


In [15]:

# ============================================================================
# SUMMARY
# ============================================================================
print("\n" + "="*70)
print("SEASONALITY SUMMARY")
print("="*70)

print("\n1. ACF PLOT FINDINGS:")


print("\n2. AUTOCORRELATION AT KEY LAGS:")


print("\n3. VISUAL AGGREGATION:")


print("\n4. PERIODOGRAM:")




SEASONALITY SUMMARY

1. ACF PLOT FINDINGS:

2. AUTOCORRELATION AT KEY LAGS:

3. VISUAL AGGREGATION:

4. PERIODOGRAM:


In [16]:
8465+295

8760

In [17]:
print(df.columns)

Index(['timestamp', 'cnt', 't1', 't2', 'hum', 'wind_speed', 'weather_code',
       'is_holiday', 'is_weekend', 'season', 'temperature_c',
       'humidity_percent', 'dew_point_c', 'rainfall_mm', 'snowfall_cm',
       'wind_speed_ms', 'solar_radiation_wm2', 'solar_radiation_mjm2'],
      dtype='object')
